In [ ]:
import cv2
import numpy as np
import os
import urllib.request
import ssl

# Stronger SSL fix — build a custom opener with SSL verification disabled
ssl_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

opener = urllib.request.build_opener(
    urllib.request.HTTPSHandler(context=ssl_context)
)
urllib.request.install_opener(opener)

# 1. Path to your screenshot image
image_path = "face_detection.jpg"
img = cv2.imread(image_path)

if img is None:
    print("Error: Could not load the image. Check your path.")
else:
    h, w = img.shape[:2]

    proto_url = "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt"
    model_url = "https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel"

    if os.path.exists("deploy.prototxt") and os.path.getsize("deploy.prototxt") == 0:
        os.remove("deploy.prototxt")

    if not os.path.exists("deploy.prototxt"):
        print("Downloading configuration file...")
        urllib.request.urlretrieve(proto_url, "deploy.prototxt")

    if not os.path.exists("res10_300x300_ssd_iter_140000.caffemodel"):
        print("Downloading model weights (~10MB)...")
        urllib.request.urlretrieve(model_url, "res10_300x300_ssd_iter_140000.caffemodel")

    net = cv2.dnn.readNetFromCaffe("deploy.prototxt", "res10_300x300_ssd_iter_140000.caffemodel")

    blob = cv2.dnn.blobFromImage(cv2.resize(img, (300, 300)), 1.0, (300, 300), (104.0, 177.0, 123.0))
    net.setInput(blob)
    detections = net.forward()

    face_count = 0
    for i in range(0, detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        if confidence > 0.5:
            face_count += 1
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            (startX, startY, endX, endY) = box.astype("int")
            cv2.rectangle(img, (startX, startY), (endX, endY), (0, 255, 0), 2)

    print(f"Total faces detected using Deep Learning: {face_count}")

    cv2.imshow("DNN Face Detection", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

Total faces detected using Deep Learning: 6
